# IEP-1003 Day 3 — ChromaDB 저장 + 스모크 테스트

**전제 조건**: IEP-1003 Day 1+2 산출물이 Drive에 존재해야 합니다.
- `IEP_1003/ipcc_chunks_semantic_p95.json` ✅

**오늘 목표**
1. p95 청크 후처리 — 5,000자 초과 청크 재분할 (llama3.1:8b 컨텍스트 보호)
2. ChromaDB 저장
3. 스모크 테스트 4종 통과 확인

| 스모크 테스트 | 기준 |
| :--- | :--- |
| ① 청크 수 확인 | ChromaDB count == 후처리 청크 수 |
| ② Retrieval 결과 | top-3 반환, 내용 관련성 있음 |
| ③ 유사도 점수 | 관련 질의 cosine distance < 0.75 |
| ④ 검색 분별력 | 무관 질의 score > 관련 질의 score |

## Step 0 — Config

In [ ]:
SOURCE_BASE  = "/content/drive/MyDrive/IPCC_data/IEP_1003"
OUTPUT_BASE  = "/content/drive/MyDrive/IPCC_data/IEP_1003"

CHUNKS_JSON     = f"{SOURCE_BASE}/ipcc_chunks_semantic_p95.json"
CHROMA_DIR      = f"{OUTPUT_BASE}/chroma_db"
COLLECTION_NAME = "ipcc_semantic_p95_v1"

EMBED_MODEL  = "jhgan/ko-sroberta-multitask"

# 후처리: 이 길이 초과 청크는 RecursiveCharacterTextSplitter로 재분할
MAX_CHUNK_SIZE = 5000
SPLIT_OVERLAP  = 200

# 스모크 테스트용 질의
SMOKE_QUERY_RELEVANT   = "지구 온난화로 인한 해수면 상승의 주요 원인은 무엇인가?"
SMOKE_QUERY_IRRELEVANT = "2024년 한국 프로야구 우승팀은 어디인가?"
SMOKE_TOP_K = 3
# jhgan/ko-sroberta-multitask + 한글 문서 기준 적정 임계값
# cosine distance 0.5~0.7 = similarity 0.3~0.5 수준으로 정상 범위
SMOKE_SCORE_THRESHOLD = 0.75

print("✅ Config 완료")

## Step 1 — 라이브러리 설치

In [ ]:
!pip install -qU chromadb langchain-community langchain-huggingface langchain-text-splitters
print("✅ 설치 완료")

## Step 2 — Drive 마운트

In [ ]:
from google.colab import drive
import os, json

drive.mount('/content/drive')
os.makedirs(CHROMA_DIR, exist_ok=True)
print("✅ Drive 마운트 완료")

## Step 3 — p95 청크 로드 + 후처리

Day 1+2 실험 결과:
- p95 최대 청크 크기: **27,236자**
- llama3.1:8b 실질 컨텍스트 한계: 약 6,000자 (한글 기준)

5,000자 초과 청크를 RecursiveCharacterTextSplitter로 재분할하여 NaN 발생을 사전에 차단합니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open(CHUNKS_JSON, "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

print(f"원본 청크 수: {len(raw_chunks)}")
print(f"5,000자 초과: {sum(1 for c in raw_chunks if c['size'] > MAX_CHUNK_SIZE)}개")
print(f"최대 크기   : {max(c['size'] for c in raw_chunks):,}자")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=MAX_CHUNK_SIZE,
    chunk_overlap=SPLIT_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""],
)

processed_chunks = []
split_count = 0

for chunk in raw_chunks:
    if chunk["size"] > MAX_CHUNK_SIZE:
        sub_texts = splitter.split_text(chunk["text"])
        for j, sub in enumerate(sub_texts):
            processed_chunks.append({
                "chunk_id"  : f"{chunk['chunk_id']}_sub{j:02d}",
                "text"      : sub,
                "size"      : len(sub),
                "threshold" : chunk["threshold"],
                "split"     : True,
            })
        split_count += 1
    else:
        processed_chunks.append({**chunk, "split": False})

sizes = [c["size"] for c in processed_chunks]
print(f"✅ 후처리 완료")
print(f"   원본 청크 수  : {len(raw_chunks)}개")
print(f"   재분할 청크 수: {split_count}개")
print(f"   최종 청크 수  : {len(processed_chunks)}개")
print(f"   평균 크기     : {int(sum(sizes)/len(sizes)):,}자")
print(f"   최대 크기     : {max(sizes):,}자")
print(f"   5,000자 초과  : {sum(1 for s in sizes if s > MAX_CHUNK_SIZE)}개 (0이어야 정상)")

In [ ]:
processed_path = f"{OUTPUT_BASE}/ipcc_chunks_semantic_p95_processed.json"
with open(processed_path, "w", encoding="utf-8") as f:
    json.dump(processed_chunks, f, ensure_ascii=False, indent=2)
print(f"✅ 후처리 청크 저장: {processed_path}")

## Step 4 — 임베딩 모델 로드

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✅ 임베딩 모델 로드 완료: {EMBED_MODEL}")

## Step 5 — ChromaDB 저장

> ⚠️ 청크 수에 따라 10~20분 소요될 수 있습니다.  
> 기존 컬렉션이 있으면 삭제 후 재생성합니다.

In [ ]:
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from tqdm.notebook import tqdm

# 기존 컬렉션 초기화
client = chromadb.PersistentClient(path=CHROMA_DIR)
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"기존 컬렉션 삭제: {COLLECTION_NAME}")

# Document 객체 변환
docs = [
    Document(
        page_content=c["text"],
        metadata={
            "chunk_id" : c["chunk_id"],
            "size"     : c["size"],
            "threshold": c["threshold"],
            "split"    : str(c["split"]),
        }
    )
    for c in processed_chunks
]

# 배치 저장 (T4 메모리 안전)
BATCH_SIZE = 50
vectorstore = None

for i in tqdm(range(0, len(docs), BATCH_SIZE), desc="ChromaDB 저장"):
    batch = docs[i:i + BATCH_SIZE]
    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=CHROMA_DIR,
        )
    else:
        vectorstore.add_documents(batch)

count = vectorstore._collection.count()
print(f"✅ ChromaDB 저장 완료")
print(f"   컬렉션    : {COLLECTION_NAME}")
print(f"   저장 청크 : {count}개")
assert count == len(processed_chunks), f"청크 수 불일치: DB={count}, 원본={len(processed_chunks)}"

## Step 6 — 스모크 테스트

4종 테스트를 순서대로 실행합니다. 모두 PASS여야 RAGAS 평가로 진입합니다.

In [ ]:
smoke_results = {}

def smoke_pass(name):
    smoke_results[name] = "PASS"
    print(f"  ✅ PASS: {name}")

def smoke_fail(name, reason):
    smoke_results[name] = f"FAIL: {reason}"
    print(f"  ❌ FAIL: {name} — {reason}")

print("=== 스모크 테스트 시작 ===")

In [ ]:
# ── 테스트 ① : 청크 수 확인 ───────────────────────────────────────
print("\n[테스트 ①] ChromaDB 청크 수 확인")
db_count = vectorstore._collection.count()
expected = len(processed_chunks)
print(f"  DB 청크 수: {db_count} / 기대값: {expected}")

if db_count == expected:
    smoke_pass("① 청크 수 일치")
else:
    smoke_fail("① 청크 수 일치", f"DB={db_count}, 기대={expected}")

In [ ]:
# ── 테스트 ② : 관련 질의 retrieval 결과 비어있지 않음 ───────────────
print("\n[테스트 ②] 관련 질의 Retrieval 결과 확인")
print(f"  질의: {SMOKE_QUERY_RELEVANT}")

retriever = vectorstore.as_retriever(search_kwargs={"k": SMOKE_TOP_K})
rel_docs = retriever.invoke(SMOKE_QUERY_RELEVANT)

print(f"  반환 청크 수: {len(rel_docs)}")
for i, doc in enumerate(rel_docs):
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"    [{i+1}] ({len(doc.page_content)}자) {preview}...")

if len(rel_docs) == SMOKE_TOP_K:
    smoke_pass("② Retrieval 결과 비어있지 않음")
else:
    smoke_fail("② Retrieval 결과 비어있지 않음", f"반환={len(rel_docs)}, 기대={SMOKE_TOP_K}")

In [ ]:
# ── 테스트 ③ : 관련 질의 유사도 점수 확인 ──────────────────────────
print("\n[테스트 ③] 관련 질의 유사도 점수 확인")
print(f"  질의: {SMOKE_QUERY_RELEVANT}")
print(f"  기준: cosine distance < {SMOKE_SCORE_THRESHOLD} (낮을수록 유사)")

rel_results = vectorstore.similarity_search_with_score(SMOKE_QUERY_RELEVANT, k=SMOKE_TOP_K)
rel_scores = [score for _, score in rel_results]

for i, (doc, score) in enumerate(rel_results):
    preview = doc.page_content[:60].replace("\n", " ")
    print(f"    [{i+1}] score={score:.4f}  {preview}...")

if all(s < SMOKE_SCORE_THRESHOLD for s in rel_scores):
    smoke_pass("③ 관련 질의 유사도 점수 정상")
else:
    over = [round(s, 4) for s in rel_scores if s >= SMOKE_SCORE_THRESHOLD]
    smoke_fail("③ 관련 질의 유사도 점수 정상", f"{SMOKE_SCORE_THRESHOLD} 초과: {over}")

In [ ]:
# ── 테스트 ④ : 무관 질의 score > 관련 질의 score (검색 분별력) ──────
print("\n[테스트 ④] 무관 질의 유사도 비교 (검색 분별력 확인)")
print(f"  무관 질의: {SMOKE_QUERY_IRRELEVANT}")

irr_results = vectorstore.similarity_search_with_score(SMOKE_QUERY_IRRELEVANT, k=SMOKE_TOP_K)
irr_scores = [score for _, score in irr_results]

for i, (doc, score) in enumerate(irr_results):
    preview = doc.page_content[:60].replace("\n", " ")
    print(f"    [{i+1}] score={score:.4f}  {preview}...")

rel_mean = sum(rel_scores) / len(rel_scores)
irr_mean = sum(irr_scores) / len(irr_scores)
print(f"\n  관련 질의 평균 score : {rel_mean:.4f}")
print(f"  무관 질의 평균 score : {irr_mean:.4f}")

if irr_mean > rel_mean:
    smoke_pass("④ 무관 질의 score > 관련 질의 score")
else:
    smoke_fail("④ 무관 질의 score > 관련 질의 score", f"무관({irr_mean:.4f}) <= 관련({rel_mean:.4f})")

## Step 7 — 스모크 테스트 최종 결과

In [ ]:
print("\n" + "=" * 45)
print("     스모크 테스트 최종 결과")
print("=" * 45)

all_pass = True
for name, result in smoke_results.items():
    icon = "✅" if result == "PASS" else "❌"
    print(f"  {icon}  {name}: {result}")
    if result != "PASS":
        all_pass = False

print("=" * 45)

if all_pass:
    print("\n✅ 전체 PASS — RAGAS 평가 진입 가능")
    print(f"   컬렉션    : {COLLECTION_NAME}")
    print(f"   ChromaDB  : {CHROMA_DIR}")
    print(f"   최종 청크 : {len(processed_chunks)}개")
    print("\n  ▶ 다음 단계: IEP-1003 Day 4 — RAGAS 4지표 평가")
else:
    print("\n❌ FAIL 항목이 있습니다. 위 로그를 확인하고 수정 후 재실행하세요.")